In [ ]:
import uproot
import matplotlib.pyplot as plt
import numpy as np

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
events = []
with uproot.open("../track_data.root") as fdata:
    arrays = fdata["selected_data/hits"].arrays(library="np")
    event_ids = np.unique(arrays['event_id'])
    for eid in event_ids:
        events.append({ k : v[arrays['event_id'] == eid] for k, v in arrays.items()})
    # print(fdata["selected_data/hits"].keys())

In [ ]:
highq = 10
def analyze_event(ievent):
    event = events[ievent]
    for j in range(8):
        hits = {k : v[event['io_group'] == j] for k, v in event.items()}
        if len(hits['Q']) == 0:
            continue
    
        highq_hits = {k : v[hits['totQ'] > highq] for k, v in hits.items()}
        if len(highq_hits['Q']) == 0:
            raise ValueError

        unique_highq_hits = {}
        for ih, q in enumerate(highq_hits['Q']):
            xyzq = np.array([highq_hits['x'][ih], highq_hits['y'][ih], highq_hits['z'][ih], highq_hits['Q'][ih]])
            yzint = tuple(np.rint(xyzq[[1,2]]*1000).astype(int).tolist())
            q = unique_highq_hits.get(yzint, [0,0,0,0])[-1]
            # larger q is kept
            if q < xyzq[-1]:
                unique_highq_hits[yzint] = xyzq
        # print(unique_highq_hits)
        print(len(unique_highq_hits))

        xyzq = np.array([i.tolist() for i in unique_highq_hits.values()])
        centroid = np.average(xyzq[:, :-1], axis=0)
        xyz_recenter = xyzq[:,:-1] - centroid[None,:]
        pca = PCA(n_components=1)
        xyz_pca = pca.fit_transform(xyz_recenter)
        direction = pca.components_[0]

        print(np.max(np.dot(xyz_recenter, direction)))

        line = centroid + np.linspace(-30, 30, 200)[:,None] * direction[None,:]
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(*xyzq[:,:-1].T, c=xyzq[:,-1], alpha=0.4, label="hits")
        ax.plot(*line.T, 'b--', label="PCA")
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
        ax.legend()
        ax.set_title(f"io_group {j}")
        plt.tight_layout()
        plt.show()
        print(direction)


        fig2, axes = plt.subplots(1, 3, figsize=(18,6))
        axes[0].scatter(xyzq[:,0], xyzq[:,1], label='selected hits')
        axes[0].plot(line[:,0], line[:,1], 'r-', label='PCA')
        
        axes[1].scatter(xyzq[:,1], xyzq[:,2], label='selected hits')
        axes[1].plot(line[:,1], line[:,2], 'r-', label='PCA')

        axes[2].scatter(xyzq[:,2], xyzq[:,0], label='selected hits')
        axes[2].plot(line[:,2], line[:,0], 'r-', label='PCA')


        hits_xyz = np.column_stack([hits['x'], hits['y'], hits['z']])
        hits_xyz_recenter = hits_xyz - centroid
        proj = np.dot(hits_xyz_recenter, direction)

        # Step 2: Define variable bin edges every 2 units
        bin_width = 2
        proj_min = proj.min()
        proj_max = proj.max()

        # Ensure final bin includes max even if it?s < 2 wide
        bin_edges = np.arange(proj_min, proj_max + bin_width, bin_width)
        if bin_edges[-1] < proj_max:
            bin_edges = np.append(bin_edges, proj_max)

        # Step 3: Compute weighted histogram (sum of 'Q' per bin)
        hist, _ = np.histogram(proj, bins=bin_edges, weights=hits['Q'])

        # Step 4: Normalize by bin width
        bin_widths = np.diff(bin_edges)
        hist_normalized = hist / bin_widths  # now it?s like dQ/ds

        # Step 5: Plot
        bin_centers = bin_edges[:-1] + bin_widths / 2

        fig = plt.figure()
        plt.bar(bin_centers, hist_normalized, width=bin_widths, align='center', edgecolor='k')
        plt.xlabel("Projected Distance (along fit)")
        plt.ylabel("Q Sum / Length (dQ/ds)")
        plt.title("Charge Profile Along Track")
        plt.grid(True)
        plt.tight_layout()
        fig.savefig(f"event_id_{event['event_id'][0]}_entry{ievent}_io_group{j}_dQdx.png")

        # perform a weighted        

In [ ]:
analyze_event(0)

In [ ]:
analyze_event(1)

In [ ]:
analyze_event(2)

In [ ]:
for i in range(len(events)):
    analyze_event(i)